In [1]:
import dotenv
import openai
import requests
import json
from dotenv import load_dotenv

load_dotenv()
client = openai.OpenAI()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    res = requests.get(f"{BASE_URL}/movies")
    return res.json()

def get_movie_details(movie_id):
    res = requests.get(f"{BASE_URL}/movies/{movie_id}")
    return res.json()

def get_movie_credits(movie_id):
    res = requests.get(f"{BASE_URL}/movies/{movie_id}/credits")
    return res.json()

# 함수 이름 → 실제 함수 매핑
FUNCTION_MAP = {
    "get_popular_movies": lambda args: get_popular_movies(),
    "get_movie_details": lambda args: get_movie_details(args["id"]),
    "get_movie_credits": lambda args: get_movie_credits(args["id"]),
}

PROMPT_TEMPLATE = """I have the following functions in my system.

`get_popular_movies()` - /movies 에서 인기 영화 목록을 가져옵니다. 인자 없음.
`get_movie_details(id)` - /movies/:id 에서 영화 상세 정보를 가져옵니다. 영화 ID를 인자로 받습니다.
`get_movie_credits(id)` - /movies/:id/credits 에서 출연진 및 제작진을 가져옵니다. 영화 ID를 인자로 받습니다.

Recommendation rule:
- If the user has already watched a movie before, do not recommend that movie again.
- Prefer recommending movies that the user has not seen yet.
- If the user mentions a movie they have seen, exclude it from future recommendations.

Please answer with the name of function that you would like me to run.
If the function requires arguments, include them like: get_movie_details(550)

Please say nothing else, just the name of the function with the arguments.

Answer the following question:

{question}
"""

dotenv.load_dotenv()

client = openai.OpenAI()

In [2]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 영화 목록을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 상세 정보(줄거리, 평점, 개봉일 등)를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID"}
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화 출연진 및 제작진 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID"}
                },
                "required": ["id"],
            },
        },
    },
]

def build_system_prompt(user_profile: dict) -> str:
    genres = ", ".join(user_profile["favorite_genres"]) if user_profile["favorite_genres"] else "아직 파악되지 않음"
    watched = ", ".join(user_profile["watched_movies"]) if user_profile["watched_movies"] else "없음"

    return f"""당신은 영화 추천 어시스턴트입니다.
            ## 사용자 프로필
            - 선호 장르: {genres}
            - 이미 시청한 영화: {watched}

            ## 규칙
            1. 이미 시청한 영화는 절대 추천하지 마세요.
            2. 사용자의 선호 장르를 우선적으로 고려하세요.
            3. 추천 시 왜 이 영화를 추천하는지 간단히 설명하세요.
            4. 사용자가 선호 장르나 시청 영화를 언급하면 기억하고 있음을 보여주세요.
            5. 한국어로 답변하세요.
            6. 필요할 때 tool을 사용해 최신 인기 영화 정보를 가져오세요.
            """

In [3]:
class MovieChatbot:
    def __init__(self):
        self.messages: list[dict] = []          # 대화 기록 (memory)
        self.user_profile = {
            "favorite_genres": [],               # 선호 장르
            "watched_movies": [],                # 시청한 영화
        }

    def _get_messages_for_api(self) -> list[dict]:
        """system prompt(프로필 반영) + 대화 기록을 합쳐서 반환"""
        system_msg = {"role": "system", "content": build_system_prompt(self.user_profile)}
        return [system_msg] + self.messages

    def _execute_tool_calls(self, tool_calls) -> None:
        """tool_calls를 실행하고 결과를 messages에 추가"""
        for tc in tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

            if fn_name in FUNCTION_MAP:
                result = FUNCTION_MAP[fn_name](fn_args)
            else:
                result = {"error": f"Unknown function: {fn_name}"}

            self.messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "name": fn_name,
                "content": json.dumps(result, ensure_ascii=False),
            })

    def chat(self, user_input: str) -> str:
        """
        사용자 입력을 받아 응답을 반환.
        tool_calls가 있으면 실행 후 재호출 (loop).
        """
        # 1. 사용자 메시지 저장
        self.messages.append({"role": "user", "content": user_input})

        # 2. API 호출 루프
        while True:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                tools=TOOLS,
                messages=self._get_messages_for_api(),
            )

            choice = response.choices[0]
            message = choice.message

            # assistant 응답 저장
            msg_dict = {"role": "assistant", "content": message.content}
            if message.tool_calls:
                msg_dict["tool_calls"] = [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in message.tool_calls
                ]
            self.messages.append(msg_dict)

            # tool_calls가 있으면 실행 후 재호출
            if choice.finish_reason == "tool_calls":
                self._execute_tool_calls(message.tool_calls)
                continue

            # 최종 응답 반환
            return message.content or ""

In [4]:
def send(bot: MovieChatbot, user_input: str):
    """메시지 전송 + 출력을 한 줄로"""
    print(f"User: {user_input}")
    response = bot.chat(user_input)
    print(f"AI: {response}")
    print()
    return response

In [6]:
bot = MovieChatbot()

send(bot, "나는 코미디 영화를 좋아해")




User: 나는 코미디 영화를 좋아해
AI: 다음은 현재 인기 있는 코미디 영화 몇 편입니다.

1. **[Mike & Nick & Nick & Alice](https://www.themoviedb.org/movie/1115544-mike-nick-nick-alice)** (2026-03-14)
   - ![포스터](https://image.tmdb.org/t/p/w780/7F0jc75HrSkLVcvOXR2FXAIwuEv.jpg)
   - 두 갱스터와 그들이 사랑하는 여자가 가장 위험한 밤을 생존하려고 하는 이야기입니다. 그날에 시간 기계가 포함되어 있어 더욱 흥미로운 요소가 추가됩니다.

2. **[GOAT](https://www.themoviedb.org/movie/1297842-goat)** (2026-02-11)
   - ![포스터](https://image.tmdb.org/t/p/w780/wfuqMlaExcoYiUEvKfVpUTt1v4u.jpg)
   - 한 작은 염소가 프로 스포츠에 참가해 생존을 위해 경쟁하는 이야기로, 유머와 감동을 동시에 느낄 수 있습니다.

3. **[The Super Mario Galaxy Movie](https://www.themoviedb.org/movie/1226863-the-super-mario-galaxy-movie)** (2026-04-01)
   - ![포스터](https://image.tmdb.org/t/p/w780/eJGWx219ZcEMVQJhAgMiqo8tYY.jpg)
   - 마리오와 루이지 형제가 새로운 악당과 맞서 싸우는 모험을 그린 애니메이션으로, 가족 모두가 즐길 수 있는 유쾌한 이야기입니다.

각 영화는 코미디 요소와 흥미로운 줄거리가 있어 즐겁게 시청하실 수 있을 것입니다! 무엇이든 궁금한 점이 있다면 말씀해 주세요.



'다음은 현재 인기 있는 코미디 영화 몇 편입니다.\n\n1. **[Mike & Nick & Nick & Alice](https://www.themoviedb.org/movie/1115544-mike-nick-nick-alice)** (2026-03-14)\n   - ![포스터](https://image.tmdb.org/t/p/w780/7F0jc75HrSkLVcvOXR2FXAIwuEv.jpg)\n   - 두 갱스터와 그들이 사랑하는 여자가 가장 위험한 밤을 생존하려고 하는 이야기입니다. 그날에 시간 기계가 포함되어 있어 더욱 흥미로운 요소가 추가됩니다.\n\n2. **[GOAT](https://www.themoviedb.org/movie/1297842-goat)** (2026-02-11)\n   - ![포스터](https://image.tmdb.org/t/p/w780/wfuqMlaExcoYiUEvKfVpUTt1v4u.jpg)\n   - 한 작은 염소가 프로 스포츠에 참가해 생존을 위해 경쟁하는 이야기로, 유머와 감동을 동시에 느낄 수 있습니다.\n\n3. **[The Super Mario Galaxy Movie](https://www.themoviedb.org/movie/1226863-the-super-mario-galaxy-movie)** (2026-04-01)\n   - ![포스터](https://image.tmdb.org/t/p/w780/eJGWx219ZcEMVQJhAgMiqo8tYY.jpg)\n   - 마리오와 루이지 형제가 새로운 악당과 맞서 싸우는 모험을 그린 애니메이션으로, 가족 모두가 즐길 수 있는 유쾌한 이야기입니다.\n\n각 영화는 코미디 요소와 흥미로운 줄거리가 있어 즐겁게 시청하실 수 있을 것입니다! 무엇이든 궁금한 점이 있다면 말씀해 주세요.'

In [7]:
send(bot, "가문의 영광 재미있나?")

User: 가문의 영광 재미있나?
AI: "가문의 영광"은 한국의 코미디 영화로, 가족 간의 갈등과 화해를 다루고 있습니다. 주로 유머와 감동을 잘 결합한 작품으로 알려져 있어 많은 관객들에게 사랑받았습니다. 이 영화는 가족과의 유대감을 느끼게 해주고, 여러 세대의 갈등을 코믹하게 풀어낸 장면들이 매력적입니다. 

만약 한국 코미디를 좋아하신다면, "가문의 영광"도 재미있게 보실 수 있을 것 같아요! 이 영화에 대해 좀 더 궁금한 점이나 다른 추천이 필요하시다면 언제든지 말씀해 주세요.



'"가문의 영광"은 한국의 코미디 영화로, 가족 간의 갈등과 화해를 다루고 있습니다. 주로 유머와 감동을 잘 결합한 작품으로 알려져 있어 많은 관객들에게 사랑받았습니다. 이 영화는 가족과의 유대감을 느끼게 해주고, 여러 세대의 갈등을 코믹하게 풀어낸 장면들이 매력적입니다. \n\n만약 한국 코미디를 좋아하신다면, "가문의 영광"도 재미있게 보실 수 있을 것 같아요! 이 영화에 대해 좀 더 궁금한 점이나 다른 추천이 필요하시다면 언제든지 말씀해 주세요.'

In [10]:
send(bot, "두섭아 일정 많이 컸.. 잡앗다 이제")

User: 두섭아 일정 많이 컸.. 잡앗다 이제
AI: 두섭이라는 이름이 특별한 의미가 있으신가요? 혹시 "가문의 영광"을 보고 난 후 주변 사람들과 함께 즐길 수 있는 다른 영화 추천이 필요하시다면 말씀해 주세요! 지금 말씀하신 것처럼 일정을 정리하신 후에 보기에 좋은 코미디 영화들을 추천해드릴 수 있습니다.



'두섭이라는 이름이 특별한 의미가 있으신가요? 혹시 "가문의 영광"을 보고 난 후 주변 사람들과 함께 즐길 수 있는 다른 영화 추천이 필요하시다면 말씀해 주세요! 지금 말씀하신 것처럼 일정을 정리하신 후에 보기에 좋은 코미디 영화들을 추천해드릴 수 있습니다.'

In [11]:
send(bot, "나 미스터빈 봤음")

User: 나 미스터빈 봤음
AI: "미스터 빈"을 보셨다고 하니, 그 특유의 유머와 경쾌한 스타일을 좋아하시는 것 같네요! 이 스타일에 맞는 코미디 영화 중 추천해드릴게요.

1. **[GOAT](https://www.themoviedb.org/movie/1297842-goat)** (2026-02-11)
   - ![포스터](https://image.tmdb.org/t/p/w780/wfuqMlaExcoYiUEvKfVpUTt1v4u.jpg)
   - 한 작은 염소가 프로 스포츠에 참가해 생존을 위해 경쟁하는 이야기로, 유머와 감동을 동시에 느낄 수 있습니다.

2. **[The Super Mario Galaxy Movie](https://www.themoviedb.org/movie/1226863-the-super-mario-galaxy-movie)** (2026-04-01)
   - ![포스터](https://image.tmdb.org/t/p/w780/eJGWx219ZcEMVQJhAgMiqo8tYY.jpg)
   - 마리오와 루이지 형제가 새로운 악당과 맞서 싸우는 모험을 그린 애니메이션으로, 가족 모두가 즐길 수 있는 유쾌한 이야기입니다.

이 두 영화 모두 유머가 많은 작품이어서 "미스터 빈"을 좋아하신다면 즐겁게 시청하실 수 있을 것 같아요! 다른 추천이 필요하시거나 더 많은 정보를 원하시면 언제든지 말씀해 주세요.



'"미스터 빈"을 보셨다고 하니, 그 특유의 유머와 경쾌한 스타일을 좋아하시는 것 같네요! 이 스타일에 맞는 코미디 영화 중 추천해드릴게요.\n\n1. **[GOAT](https://www.themoviedb.org/movie/1297842-goat)** (2026-02-11)\n   - ![포스터](https://image.tmdb.org/t/p/w780/wfuqMlaExcoYiUEvKfVpUTt1v4u.jpg)\n   - 한 작은 염소가 프로 스포츠에 참가해 생존을 위해 경쟁하는 이야기로, 유머와 감동을 동시에 느낄 수 있습니다.\n\n2. **[The Super Mario Galaxy Movie](https://www.themoviedb.org/movie/1226863-the-super-mario-galaxy-movie)** (2026-04-01)\n   - ![포스터](https://image.tmdb.org/t/p/w780/eJGWx219ZcEMVQJhAgMiqo8tYY.jpg)\n   - 마리오와 루이지 형제가 새로운 악당과 맞서 싸우는 모험을 그린 애니메이션으로, 가족 모두가 즐길 수 있는 유쾌한 이야기입니다.\n\n이 두 영화 모두 유머가 많은 작품이어서 "미스터 빈"을 좋아하신다면 즐겁게 시청하실 수 있을 것 같아요! 다른 추천이 필요하시거나 더 많은 정보를 원하시면 언제든지 말씀해 주세요.'

In [12]:
send(bot, "요즘 코미디 어때? 볼만한 코미디 영화 있는가?")

User: 요즘 코미디 어때? 볼만한 코미디 영화 있는가?
AI: 요즘 코미디 영화들도 재미있고 화제가 많은 작품들이 있습니다. 다음은 요즘 인기 있는 코미디 영화 몇 편입니다:

1. **[Mike & Nick & Nick & Alice](https://www.themoviedb.org/movie/1115544-mike-nick-nick-alice)** (2026-03-14)
   - ![포스터](https://image.tmdb.org/t/p/w780/7F0jc75HrSkLVcvOXR2FXAIwuEv.jpg)
   - 두 갱스터가 사랑하는 여인과 함께 가장 위험한 밤을 생존하려고 애쓰는 이야기입니다. 중간에 시간 기계라는 흥미로운 요소가 추가됩니다.

2. **[GOAT](https://www.themoviedb.org/movie/1297842-goat)** (2026-02-11)
   - ![포스터](https://image.tmdb.org/t/p/w780/wfuqMlaExcoYiUEvKfVpUTt1v4u.jpg)
   - 한 염소가 프로 스포츠에 도전하는 이야기를 그린 영화로, 많은 유머와 따뜻한 메시지를 담고 있습니다.

3. **[The Super Mario Galaxy Movie](https://www.themoviedb.org/movie/1226863-the-super-mario-galaxy-movie)** (2026-04-01)
   - ![포스터](https://image.tmdb.org/t/p/w780/eJGWx219ZcEMVQJhAgMiqo8tYY.jpg)
   - 마리오와 루이지 형제가 새로운 악당에 맞서 싸우는 가족 친화적인 애니메이션으로, 유쾌한 웃음을 선사합니다.

이 영화는 재미있고 가벼운 마음으로 즐길 수 있는 작품들이므로, 편안한 시간에 감상해 보시면 좋을 것 같아요! 궁금한 점이 더 있으면 언제든지 말씀해 주세요.



'요즘 코미디 영화들도 재미있고 화제가 많은 작품들이 있습니다. 다음은 요즘 인기 있는 코미디 영화 몇 편입니다:\n\n1. **[Mike & Nick & Nick & Alice](https://www.themoviedb.org/movie/1115544-mike-nick-nick-alice)** (2026-03-14)\n   - ![포스터](https://image.tmdb.org/t/p/w780/7F0jc75HrSkLVcvOXR2FXAIwuEv.jpg)\n   - 두 갱스터가 사랑하는 여인과 함께 가장 위험한 밤을 생존하려고 애쓰는 이야기입니다. 중간에 시간 기계라는 흥미로운 요소가 추가됩니다.\n\n2. **[GOAT](https://www.themoviedb.org/movie/1297842-goat)** (2026-02-11)\n   - ![포스터](https://image.tmdb.org/t/p/w780/wfuqMlaExcoYiUEvKfVpUTt1v4u.jpg)\n   - 한 염소가 프로 스포츠에 도전하는 이야기를 그린 영화로, 많은 유머와 따뜻한 메시지를 담고 있습니다.\n\n3. **[The Super Mario Galaxy Movie](https://www.themoviedb.org/movie/1226863-the-super-mario-galaxy-movie)** (2026-04-01)\n   - ![포스터](https://image.tmdb.org/t/p/w780/eJGWx219ZcEMVQJhAgMiqo8tYY.jpg)\n   - 마리오와 루이지 형제가 새로운 악당에 맞서 싸우는 가족 친화적인 애니메이션으로, 유쾌한 웃음을 선사합니다.\n\n이 영화는 재미있고 가벼운 마음으로 즐길 수 있는 작품들이므로, 편안한 시간에 감상해 보시면 좋을 것 같아요! 궁금한 점이 더 있으면 언제든지 말씀해 주세요.'